# Continuation- The Need for Representation: A Sentiment Analysis Perspective

## 4. GloVe: Global Vectors for Word Representation

In our exploration of Word2Vec, we saw that both Continuous Bag-of-Words (CBOW) and Skip-Gram are predictive models. They learn representations by strictly looking at local context windows (e.g., one word to the left, one word to the right).

While highly effective, this local-window approach completely ignores the **global statistical properties** of the entire corpus. On the other hand, count-based models like the Term-Document Matrix capture global corpus statistics beautifully, but they fail to capture meaning and suffer from extreme sparsity.

**GloVe (Global Vectors)**, introduced by researchers at Stanford in 2014, bridges this gap. It asks: *Why not combine the global mathematical precision of count-based matrices with the dense semantic spaces of local predictive models?*

### The Co-occurrence Matrix
Instead of stepping through the corpus one window at a time during training, GloVe first builds a **Global Co-occurrence Matrix** ($X$).

Each cell $X_{ij}$ represents how many times word $j$ appears in the context of word $i$ across the *entire dataset*.

In [1]:
# ---------------------------------------------------------
# 1. Define Corpus and Build Vocabulary
# ---------------------------------------------------------
corpus = [
    "the movie was excellent",
    "the film was excellent",
    "the movie was terrible",
    "the film was terrible"
]

In [2]:
# Tokenize the corpus
tokenized_corpus = [sentence.split() for sentence in corpus]

# Build the vocabulary
vocabulary = set([word for sentence in tokenized_corpus for word in sentence])
vocab_size = len(vocabulary)

# Create mappings from word to index and index to word
word_to_ix = {word: i for i, word in enumerate(vocabulary)}
ix_to_word = {i: word for i, word in enumerate(vocabulary)}

print(f"Vocabulary Size: {vocab_size}")
print(f"Word to Index Map: {word_to_ix}")

Vocabulary Size: 6
Word to Index Map: {'movie': 0, 'was': 1, 'film': 2, 'excellent': 3, 'terrible': 4, 'the': 5}


In [3]:
# Let's build a co-occurrence matrix for our toy corpus to see what GloVe actually trains on.

import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
import pandas as pd
from collections import defaultdict
import numpy as np

In [4]:
# We will reuse the 'tokenized_corpus', 'word_to_ix', and 'vocab_size'
# from our Word2Vec implementation earlier in the notebook.

vocab = list(vocabulary)

# 1. Initialize an empty co-occurrence matrix
co_oc_matrix = defaultdict(float)
WINDOW_SIZE = 1

# 2. Populate the matrix by sliding a window across the corpus
for sentence in tokenized_corpus:
    for i, center_word in enumerate(sentence):
        for j in range(max(0, i - WINDOW_SIZE), min(len(sentence), i + WINDOW_SIZE + 1)):
            if i != j:
                context_word = sentence[j]
                # Increment the co-occurrence count
                co_oc_matrix[(word_to_ix[center_word], word_to_ix[context_word])] += 1.0

# Let's visualize this as a dense DataFrame to understand global statistics
dense_co_oc = np.zeros((vocab_size, vocab_size))
for (w_c, w_cx), count in co_oc_matrix.items():
    dense_co_oc[w_c, w_cx] = count

df_co_oc = pd.DataFrame(dense_co_oc, index=vocab, columns=vocab)
print("--- Global Word Co-occurrence Matrix (X) ---")
display(df_co_oc)

--- Global Word Co-occurrence Matrix (X) ---


,movie,was,film,excellent,terrible,the
movie,0.0,2.0,0.0,0.0,0.0,2.0
was,2.0,0.0,2.0,2.0,2.0,0.0
film,0.0,2.0,0.0,0.0,0.0,2.0
excellent,0.0,2.0,0.0,0.0,0.0,0.0
terrible,0.0,2.0,0.0,0.0,0.0,0.0
the,2.0,0.0,2.0,0.0,0.0,0.0


### The GloVe Objective Function

GloVe is essentially a weighted matrix factorization model. It forces the model to learn word vectors such that their dot product equals the logarithm of the words' probability of co-occurrence.

Mathematically, for a target word vector $w_i$, a context word vector $\tilde{w}_j$, and their respective biases $b_i$ and $\tilde{b}_j$:

$$w_i^T \tilde{w}_j + b_i + \tilde{b}_j = \log(X_{ij})$$

To train this, GloVe minimizes the following weighted Mean Squared Error (MSE) loss function:

$$J = \sum_{i,j=1}^{V} f(X_{ij}) \left( w_i^T \tilde{w}_j + b_i + \tilde{b}_j - \log(X_{ij}) \right)^2$$

Notice the weighting function $f(X_{ij})$. GloVe uses this to cap the influence of wildly frequent words (like "the" and "was") while preventing rare combinations from dominating the network.

### Under the Hood: Building GloVe from Scratch

Just like we did for Skip-Gram and CBOW, let's implement a toy version of GloVe in PyTorch.

In [5]:
# ---------------------------------------------------------
# 1. Define the GloVe Model
# ---------------------------------------------------------
class GloVeModel(nn.Module):
    def __init__(self, vocab_size, embedding_dim):
        super(GloVeModel, self).__init__()
        # GloVe maintains separate embeddings for a word when it is the center
        # versus when it is a context word.
        self.w_embeddings = nn.Embedding(vocab_size, embedding_dim)
        self.c_embeddings = nn.Embedding(vocab_size, embedding_dim)

        # It also maintains separate biases
        self.w_biases = nn.Embedding(vocab_size, 1)
        self.c_biases = nn.Embedding(vocab_size, 1)

    def forward(self, w_ix, c_ix):
        # Extract vectors
        w = self.w_embeddings(w_ix)
        c = self.c_embeddings(c_ix)
        b_w = self.w_biases(w_ix)
        b_c = self.c_biases(c_ix)

        # Compute dot product: w^T * c
        dot_product = torch.sum(w * c, dim=1, keepdim=True)

        # Return predicted log(X_ij)
        return dot_product + b_w + b_c

# ---------------------------------------------------------
# 2. GloVe Weighting Function
# ---------------------------------------------------------
def weighting_function(x, x_max=10, alpha=0.75):
    """Caps the impact of extremely frequent co-occurrences."""
    return (x / x_max)**alpha if x < x_max else 1.0

# Prepare PyTorch training data
glove_data = [(w, c, count) for (w, c), count in co_oc_matrix.items()]

# Hyperparameters
EMBEDDING_DIM = 2
EPOCHS = 200
LEARNING_RATE = 0.05

glove_model = GloVeModel(vocab_size, EMBEDDING_DIM)
optimizer_glove = optim.Adagrad(glove_model.parameters(), lr=LEARNING_RATE)
# ---------------------------------------------------------
# 3. Train the GloVe Model
# ---------------------------------------------------------
print("--- Training GloVe ---")
for epoch in range(EPOCHS):
    total_loss = 0
    for w_ix, c_ix, count in glove_data:
        # Step 1: Prepare inputs
        w_tensor = torch.tensor([w_ix], dtype=torch.long)
        c_tensor = torch.tensor([c_ix], dtype=torch.long)
        log_x_ij = torch.tensor([[np.log(count)]], dtype=torch.float)
        weight = torch.tensor([[weighting_function(count)]], dtype=torch.float)

        # Step 2: Zero gradients
        glove_model.zero_grad()

        # Step 3: Forward pass
        prediction = glove_model(w_tensor, c_tensor)

        # Step 4: Compute weighted MSE loss and backpropagate
        loss = weight * F.mse_loss(prediction, log_x_ij)
        loss.backward()
        optimizer_glove.step()

        total_loss += loss.item()

    if (epoch + 1) % 50 == 0:
        print(f"Epoch {epoch+1}/{EPOCHS} | Loss: {total_loss:.4f}")

# Final embeddings in GloVe are traditionally the sum of center and context vectors
final_glove_embeddings = glove_model.w_embeddings.weight.data + glove_model.c_embeddings.weight.data
print("\nFinal GloVe Vector for 'movie':")
print(final_glove_embeddings[word_to_ix['movie']].numpy())

--- Training GloVe ---
Epoch 50/200 | Loss: 0.4080
Epoch 100/200 | Loss: 0.1436
Epoch 150/200 | Loss: 0.0668
Epoch 200/200 | Loss: 0.0346

Final GloVe Vector for 'movie':
[0.677143   0.10343283]


### Utilizing Pre-Trained GloVe

Just as training Word2Vec on massive corpora requires heavy computational resources, the true power of GloVe is realized when we leverage pre-trained matrices. Stanford provides GloVe models trained on Wikipedia, Gigaword, and massive Twitter datasets.

Let's load a 50-dimensional GloVe model using Gensim's downloader API, just like we did with the Google News model earlier.

In [6]:
!pip install gensim

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 27.8/27.8 MB 61.3 MB/s eta 0:00:00


In [7]:
import gensim.downloader as api

print("Downloading GloVe (Wiki-Gigaword 50d)...")
glove_pretrained = api.load('glove-wiki-gigaword-50')

[==================================================] 100.0% 66.0/66.0MB downloaded


In [8]:
# Let's check how GloVe handles our synonym problem!
sim_movie_film_glove = glove_pretrained.similarity('movie', 'film')
sim_excellent_terrible_glove = glove_pretrained.similarity('excellent', 'terrible')

print("\n--- Word Similarities (Pre-Trained GloVe) ---")
print(f"Similarity between 'movie' and 'film': {sim_movie_film_glove:.4f}")
print(f"Similarity between 'excellent' and 'terrible': {sim_excellent_terrible_glove:.4f}")

print("\n--- Most similar words to 'movie' (GloVe) ---")
display(pd.DataFrame(glove_pretrained.most_similar('movie', topn=3), columns=["Word", "Cosine Similarity"]))


--- Word Similarities (Pre-Trained GloVe) ---
Similarity between 'movie' and 'film': 0.9310
Similarity between 'excellent' and 'terrible': 0.3132

--- Most similar words to 'movie' (GloVe) ---


,Word,Cosine Similarity
0,movies,0.932248
1,film,0.931010
2,films,0.893739


### What did we solve?
By explicitly encoding global occurrence counts into the loss function, GloVe effectively balances two worlds. It retains the algebraic, semantic vector space we loved in Word2Vec, but it converges faster and often performs slightly better on downstream analogy tasks because it doesn't discard the macroscopic properties of the dataset.

### What problems still persist?
Notice that despite completely changing our optimization strategy, we are still bound by the same structural limitations:

1.  **Polysemy (Multiple Meanings):** GloVe creates exactly one static vector per word. It still averages all meanings into a single, blended representation.
2.  **Out-of-Vocabulary (OOV) Words:** If a word was not in the global co-occurrence matrix, the model completely fails to process it. It cannot infer the meaning of a newly coined word, a prefix, or a typo.
3. It still cannot represent a sentence directly.

## 5. Doc2Vec: Learning Document-Level Representations

In Section 3, we solved the problem of comparing documents by averaging their Word2Vec vectors (Mean Pooling). While this works reasonably well as a baseline, it has a massive theoretical flaw: **it completely destroys word order and document structure**.

To a Mean Pooling approach, the sentence *"The movie was good, not bad"* has the exact same mathematical representation as *"The movie was bad, not good"*.

To solve this, researchers Le and Mikolov (2014) introduced **Paragraph Vectors**, commonly known as **Doc2Vec**. Instead of just learning dense vectors for words, Doc2Vec introduces a dedicated, learnable vector for the document itself.

### Architectures: PV-DM vs. PV-DBOW

Just like Word2Vec has CBOW and Skip-Gram, Doc2Vec comes in two flavors:

#### 1. Distributed Memory (PV-DM)
PV-DM is the Doc2Vec equivalent of CBOW. However, alongside the context words, it passes a **Document ID vector** into the hidden layer. The document vector acts as a memory unit that remembers what is missing from the current context.

```text
 Document ID ('Doc 0') ────┐
                           │
 Context Word 1 ('the') ───┼───► [ Aggregation Layer ] ───► Target Word
                           │       (Concat or Mean)          ('movie')
 Context Word 2 ('was') ───┘

```
#### 2. Distributed Bag of Words (PV-DBOW)
PV-DBOW is the Doc2Vec equivalent of Skip-Gram. It completely ignores the context words in the input. Instead, it forces the network to predict randomly sampled words from the document using only the Document ID.

```text
                          ┌───► Random Word 1: 'the'
                          │
 Document ID ('Doc 0') ───┼───► Random Word 2: 'movie'
                          │
                          └───► Random Word 3: 'excellent'
```       

### Under the Hood: Building PV-DM from Scratch
Let's build a toy implementation of the Distributed Memory (PV-DM) architecture in PyTorch to see exactly how document vectors are learned alongside word vectors.

In [9]:
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F

In [10]:
# 1. Define Corpus and Vocabularies
corpus = [
    "the movie was excellent",      # Doc 0
    "the film was excellent",       # Doc 1
    "the movie was terrible",       # Doc 2
    "a terrible terrible movie"     # Doc 3
]

In [11]:
tokenized_corpus = [sentence.split() for sentence in corpus]
word_vocab = list(set(word for sentence in tokenized_corpus for word in sentence))
word_to_ix = {w: i for i, w in enumerate(word_vocab)}
vocab_size = len(word_vocab)

# Doc2Vec specific: We need a vocabulary for Document IDs!
doc_vocab_size = len(corpus)

In [12]:
# 2. Generate PV-DM Training Data
# We will use the previous 2 words as context to predict the target word.
CONTEXT_SIZE = 2
pvdm_data = []

for doc_id, sentence in enumerate(tokenized_corpus):
    for i in range(CONTEXT_SIZE, len(sentence)):
        context_words = [sentence[i-2], sentence[i-1]]
        target_word = sentence[i]
        pvdm_data.append((doc_id, context_words, target_word))

print(f"First 3 PV-DM Pairs (DocID, Contexts, Target): {pvdm_data[:3]}")

First 3 PV-DM Pairs (DocID, Contexts, Target): [(0, ['the', 'movie'], 'was'), (0, ['movie', 'was'], 'excellent'), (1, ['the', 'film'], 'was')]


In [13]:
# 3. Define the PV-DM Model
class PVDMModel(nn.Module):
    def __init__(self, vocab_size, doc_vocab_size, embedding_dim):
        super(PVDMModel, self).__init__()
        # We maintain TWO embedding matrices now!
        self.word_embeddings = nn.Embedding(vocab_size, embedding_dim)
        self.doc_embeddings = nn.Embedding(doc_vocab_size, embedding_dim)

        # Linear layer projects the aggregated vector to vocabulary size
        self.linear = nn.Linear(embedding_dim, vocab_size)

    def forward(self, doc_ix, context_word_indices):
        # Extract embeddings
        doc_embed = self.doc_embeddings(doc_ix)
        word_embeds = self.word_embeddings(context_word_indices)

        # Aggregate: Average the word vectors and the document vector
        # This combines local context with global document memory
        combined_embeds = torch.cat([doc_embed.unsqueeze(1), word_embeds], dim=1)
        aggregated = torch.mean(combined_embeds, dim=1)

        # Predict target word
        out = self.linear(aggregated)
        return F.log_softmax(out, dim=1)

In [14]:
# Hyperparameters
EMBEDDING_DIM = 4
EPOCHS = 300

pvdm_model = PVDMModel(vocab_size, doc_vocab_size, EMBEDDING_DIM)
loss_function = nn.NLLLoss()
optimizer = optim.SGD(pvdm_model.parameters(), lr=0.05)

In [16]:
# ---------------------------------------------------------
# 3. Define the Corrected PV-DM Model
# ---------------------------------------------------------
class PVDMModel(nn.Module):
    def __init__(self, vocab_size, doc_vocab_size, embedding_dim):
        super(PVDMModel, self).__init__()
        # Two embedding matrices: one for words, one for documents
        self.word_embeddings = nn.Embedding(vocab_size, embedding_dim)
        self.doc_embeddings = nn.Embedding(doc_vocab_size, embedding_dim)

        # Linear layer projects the aggregated vector back to vocabulary size
        self.linear = nn.Linear(embedding_dim, vocab_size)

    def forward(self, doc_ix, context_word_indices):
        # doc_embed: shape (1, embedding_dim)
        doc_embed = self.doc_embeddings(doc_ix)
        # word_embeds: shape (2, embedding_dim)
        word_embeds = self.word_embeddings(context_word_indices)

        # 1. Stack all vectors together along dim=0 -> shape (3, embedding_dim)
        combined_embeds = torch.cat([doc_embed, word_embeds], dim=0)

        # 2. Average across the vectors -> shape (1, embedding_dim)
        aggregated = torch.mean(combined_embeds, dim=0, keepdim=True)

        # 3. Project to vocabulary space -> shape (1, vocab_size)
        out = self.linear(aggregated)
        return F.log_softmax(out, dim=1)

# Hyperparameters
EMBEDDING_DIM = 4
EPOCHS = 300

pvdm_model = PVDMModel(vocab_size, doc_vocab_size, EMBEDDING_DIM)
loss_function = nn.NLLLoss()
optimizer = optim.SGD(pvdm_model.parameters(), lr=0.05)

# ---------------------------------------------------------
# 4. Train the PV-DM Model
# ---------------------------------------------------------
print("--- Training Toy PV-DM ---")
for epoch in range(EPOCHS):
    total_loss = 0
    for doc_id, context, target in pvdm_data:
        doc_tensor = torch.tensor([doc_id], dtype=torch.long)
        context_tensor = torch.tensor([word_to_ix[w] for w in context], dtype=torch.long)
        target_tensor = torch.tensor([word_to_ix[target]], dtype=torch.long)

        pvdm_model.zero_grad()
        log_probs = pvdm_model(doc_tensor, context_tensor)
        loss = loss_function(log_probs, target_tensor)
        loss.backward()
        optimizer.step()

        total_loss += loss.item()

    if (epoch + 1) % 50 == 0:
        print(f"Epoch {epoch+1}/{EPOCHS} | Loss: {total_loss:.4f}")

# The document vectors are stored in the doc_embeddings matrix!
print(f"\nLearned Document Vector for Doc 0:\n{pvdm_model.doc_embeddings.weight[0].detach().numpy().round(4)}")

--- Training Toy PV-DM ---
Epoch 50/300 | Loss: 2.9429
Epoch 100/300 | Loss: 0.8173
Epoch 150/300 | Loss: 0.3651
Epoch 200/300 | Loss: 0.2180
Epoch 250/300 | Loss: 0.1504
Epoch 300/300 | Loss: 0.1128

Learned Document Vector for Doc 0:
[ 2.3735  0.5642 -1.2849 -1.0812]


### Utilizing Gensim for Doc2Vec

In the real world, generating document embeddings using `gensim` is the standard. Unlike Word2Vec, where we often download pre-trained models, **Doc2Vec models are usually trained directly on your specific dataset** because document representations are highly dependent on the unique structures and lengths of your text.

Let's train a real Doc2Vec model on our corpus using Gensim and check the similarities.

In [17]:
from gensim.models.doc2vec import Doc2Vec, TaggedDocument
import pandas as pd
from sklearn.metrics.pairwise import cosine_similarity

# 1. Prepare data for Gensim
# Gensim requires documents to be in a specific 'TaggedDocument' format
tagged_data = [TaggedDocument(words=doc.split(), tags=[str(i)]) for i, doc in enumerate(corpus)]

# 2. Initialize and Train the Doc2Vec Model
# vector_size: Dimensionality of the document vectors
# window: Context window size
# min_count: Ignores all words with total frequency lower than this
# dm=1: Use PV-DM architecture (dm=0 uses PV-DBOW)
d2v_model = Doc2Vec(vector_size=10, window=2, min_count=1, workers=1, dm=1, epochs=100)
d2v_model.build_vocab(tagged_data)
d2v_model.train(tagged_data, total_examples=d2v_model.corpus_count, epochs=d2v_model.epochs)

# 3. Extract Document Vectors
d2v_doc_vectors = np.array([d2v_model.dv[str(i)] for i in range(len(corpus))])

# 4. Compute Document-Level Similarity Matrix
d2v_sim_matrix = cosine_similarity(d2v_doc_vectors)

# Display the results
doc_labels = [f"Doc {i}" for i in range(len(corpus))]
d2v_sim_df = pd.DataFrame(d2v_sim_matrix, index=doc_labels, columns=doc_labels)

print("--- Document Similarities (Gensim Doc2Vec) ---")
display(d2v_sim_df)

# We can also dynamically infer a vector for a completely new, unseen document!
unseen_doc = "the film was great"
unseen_vector = d2v_model.infer_vector(unseen_doc.split())
print(f"\nInferred vector for '{unseen_doc}':\n{unseen_vector.round(3)}")

--- Document Similarities (Gensim Doc2Vec) ---


,Doc 0,Doc 1,Doc 2,Doc 3
Doc 0,1.000000,0.042836,0.403129,0.210265
Doc 1,0.042836,1.000000,-0.050145,0.195751
Doc 2,0.403129,-0.050145,1.000000,0.253640
Doc 3,0.210265,0.195751,0.253640,1.000000



Inferred vector for 'the film was great':
[-0.013  0.016  0.026  0.043  0.008 -0.047 -0.016  0.004 -0.035  0.035]


### What did we solve?
1. **Document-Level Native Semantics:** Instead of crudely averaging words (which dilutes meaning), Doc2Vec explicitly optimizes a dense vector to represent the *entire* document's intent.
2. **Preservation of Order (PV-DM):** Because the Distributed Memory architecture incorporates local context windows alongside the document ID, the model captures word ordering and syntactic structure much better than Bag of Words or Mean Pooling.
3. **Dynamic Inference:** We can freeze the model weights and use the network to optimize a vector for a brand new, unseen document (using `infer_vector`), making it highly useful for real-world classification or clustering tasks.

### What problems still persist?
While Doc2Vec bridges the gap between words and documents, the evolution of Natural Language Representation hits a roadblock here.

1.  **The Polysemy Problem Remains:** Words still only have *one* static vector in the lookup table. "Bank" means the same thing whether it's next to "river" or "money".
2.  **Context Window Limitations:** The context is strictly limited to the `WINDOW_SIZE`. If a sentence starts with "I grew up in France", and 30 words later says "I speak fluent ____", neither Word2Vec nor Doc2Vec can carry that context over such a long distance.

## 6. FastText: Solving the Out-of-Vocabulary Problem

At the end of our Word2Vec and GloVe experiments, we encountered a massive limitation: **The Out-of-Vocabulary (OOV) Problem**.

If a traditional model has never seen the word `"movielike"` during training, it cannot generate a vector for it. It completely fails on typos (e.g., `"excellant"` instead of `"excellent"`) and struggles with morphologically rich variations (like `"terrible"`, `"terribly"`, `"terribleness"`).

**FastText**, developed by Facebook AI Research (FAIR), was created specifically to solve this issue.

### The Sub-Word (n-gram) Paradigm
The main principle behind FastText is that the internal, morphological structure of a word carries crucial meaning, which traditional models ignore by treating each word as a single, indivisible token.

Instead of just learning an embedding for an entire word, FastText treats every word as a collection of **character n-grams** (subwords).

For example, let's look at how FastText processes the word `"movie"` if we set our n-gram size to 3 (tri-grams):

```text
 Word:  <movie>    (Special angle brackets denote the start and end of a full word)

 3-grams (Subwords):
 ├── <mo
 ├── mov
 ├── ovi
 ├── vie
 └── ie>
 ```

 FastText learns a dense vector for each of these subwords, as well as a vector for the full word <movie>. The final vector for the word is generated by summing up the vectors of all its component n-grams.

### Why is this revolutionary?
If the model later encounters an OOV word like "movies", it doesn't crash. It simply extracts the subwords (<mo, mov, ovi, vie, ies, es>), looks up their individual vectors from the training data, and sums them together to infer a highly accurate representation for the unknown word!

In [18]:
# ---------------------------------------------------------
# Sub-word Extraction Visualization
# ---------------------------------------------------------
def get_ngrams(word, min_n=3, max_n=6):
    """
    Simulates how FastText extracts character n-grams.
    Parameters min_n and max_n control the subword lengths.
    """
    # FastText pads words with angle brackets
    padded_word = f"<{word}>"
    ngrams = []

    # Extract subwords of varying lengths
    for n in range(min_n, max_n + 1):
        for i in range(len(padded_word) - n + 1):
            ngrams.append(padded_word[i:i+n])

    # FastText also includes the full word itself as a distinct token
    ngrams.append(padded_word)
    return ngrams

word = "excellent"
print(f"Subwords for '{word}':\n")
print(f"3-grams: {[ng for ng in get_ngrams(word, 3, 3) if len(ng)==3 and ng != f'<{word}>']}")
print(f"4-grams: {[ng for ng in get_ngrams(word, 4, 4) if len(ng)==4 and ng != f'<{word}>']}")

Subwords for 'excellent':

3-grams: ['<ex', 'exc', 'xce', 'cel', 'ell', 'lle', 'len', 'ent', 'nt>']
4-grams: ['<exc', 'exce', 'xcel', 'cell', 'elle', 'llen', 'lent', 'ent>']


### Implementing FastText with Gensim

Because training FastText requires tracking millions of subword hashes, implementing it from scratch in PyTorch for a toy corpus is overly complex. Instead, we will immediately use the industry-standard implementation in `gensim`.

We will train a FastText model on our toy corpus. FastText allows us to configure `min_n` (the minimum length of a char n-gram) and `max_n` (the maximum length). By default, `gensim` usually sets `min_n=3` and `max_n=6`.

In [19]:
from gensim.models import FastText
import pandas as pd
import numpy as np

# We reuse our tokenized_corpus from earlier sections
# corpus = ["the movie was excellent", "the film was excellent", "the movie was terrible"]
# tokenized_corpus = [doc.split() for doc in corpus]

print("--- Training FastText Model ---")
# Initialize and train FastText
# window: Context window size
# min_count: Include all words
# min_n, max_n: Our subword extraction limits
ft_model = FastText(
    sentences=tokenized_corpus,
    vector_size=10,
    window=2,
    min_count=1,
    min_n=3,
    max_n=6,
    sg=1,     # Use Skip-Gram architecture
    epochs=100
)

print("Training complete!\n")

# ---------------------------------------------------------
# Testing the Out-of-Vocabulary (OOV) Superpower
# ---------------------------------------------------------
# Neither of these words exist in our tiny 4-sentence training corpus!
typo_word = "excellant"     # Misspelling of 'excellent'
morph_word = "terribly"     # Morphological variant of 'terrible'

print(f"Is '{typo_word}' in the vocabulary? {typo_word in ft_model.wv.key_to_index}")
print(f"Is '{morph_word}' in the vocabulary? {morph_word in ft_model.wv.key_to_index}\n")

# Yet, FastText can still generate vectors for them!
print(f"Inferred vector for OOV typo '{typo_word}':\n{ft_model.wv[typo_word].round(3)}\n")

# Let's check if the inferred vectors maintain semantic similarity
sim_excellent_typo = ft_model.wv.similarity('excellent', typo_word)
sim_terrible_morph = ft_model.wv.similarity('terrible', morph_word)

print("--- OOV Similarity Results ---")
print(f"Similarity ('excellent' vs OOV '{typo_word}'): {sim_excellent_typo:.4f}")
print(f"Similarity ('terrible' vs OOV '{morph_word}'): {sim_terrible_morph:.4f}")

--- Training FastText Model ---
Training complete!

Is 'excellant' in the vocabulary? False
Is 'terribly' in the vocabulary? False

Inferred vector for OOV typo 'excellant':
[-0.009  0.003  0.007  0.007  0.008 -0.006  0.005 -0.005 -0.015 -0.001]

--- OOV Similarity Results ---
Similarity ('excellent' vs OOV 'excellant'): 0.0331
Similarity ('terrible' vs OOV 'terribly'): 0.5580


### What did we solve?
1. **The Out-of-Vocabulary Problem:** FastText can obtain vectors for OOV words by summing the vectors of their component character n-grams. As long as the model has seen some of those substrings during training, it can make a highly educated guess about the word's meaning.
2. **Robustness to Typos:** Misspellings share the vast majority of their character n-grams with the correctly spelled word. FastText automatically places typo vectors extremely close to their correct counterparts.
3. **Morphological Richness:** Languages with heavy prefixing and suffixing (like Turkish, German, or Finnish) perform remarkably well under FastText because the root words are preserved within the subword n-grams.

### What problems still persist?
With FastText, we have squeezed almost every drop of performance out of the "Static Embedding" paradigm. However, the final boss of Natural Language Representation remains undefeated:

**Deep Polysemy & Contextual Dependency**
FastText still generates a **static vector** for every word. If you input the word `"bank"`, it calculates a single vector based on `<ba`, `ban`, `ank`, `nk>`.

It has absolutely no way to alter that vector based on whether the word is used in:
1. *"I deposited money in the **bank**."*
2. *"I sat by the river **bank**."*

To solve deep contextual dependencies and polysemy, we must move away from fixed embeddings and embrace sequence modeling—specifically **Recurrent Neural Networks (RNNs)** and **Transformer Architectures (BERT)**.